In [7]:
# ==============================================================================
# Cell 1: Setup, Paths, and NLP Dependencies
# ==============================================================================

"""
Configuración inicial para el procesamiento de lenguaje natural (NLP) de las 
transcripciones. Carga el modelo de spaCy (configurable para pruebas) y 
las librerías necesarias para el análisis léxico-estadístico y categorial.
"""

import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import tgt
import spacy
from wordfreq import zipf_frequency

# ------------------------------------------------------------------------------
# Rutas de sistema
# ------------------------------------------------------------------------------
BASE_DIR = Path("/home/amont21/Documentos/voxelwise modeling/ds003020/derivatives/TextGrids")
CSV_EXPORT_PATH = "audit_orthographic_inventory.csv"
SAVE_CSV_AUDIT = True  # Exportará el inventario para poder analizarlo en VS Code

# ------------------------------------------------------------------------------
# Configuración del Modelo NLP (spaCy)
# ------------------------------------------------------------------------------
# Opciones recomendadas:
# - "en_core_web_sm"  : Rápido, ligero. Ideal para probar el código.
# - "en_core_web_trf" : Basado en Transformers (RoBERTa). Lento pero máxima 
#                       precisión. Usar para la extracción definitiva de la tesis.

SPACY_MODEL_NAME = "en_core_web_trf"

print(f"Cargando modelo de spaCy: {SPACY_MODEL_NAME}...")
try:
    nlp = spacy.load(SPACY_MODEL_NAME)
    print("✅ Modelo cargado exitosamente.")
except OSError:
    print(f"❌ Error: El modelo '{SPACY_MODEL_NAME}' no está instalado.")
    print(f"Por favor, ejecuta en tu terminal: python -m spacy download {SPACY_MODEL_NAME}")

print('\nModelos disponibles en spaCy:')
# Cargar el modelo pequeño
nlp_sm = spacy.load("en_core_web_sm")
version_sm = nlp_sm.meta["version"]
print(f"Modelo SM ({version_sm}) cargado con éxito. Componentes: {nlp_sm.pipe_names}")

# Cargar el modelo Transformer
nlp_trf = spacy.load("en_core_web_trf")
version_trf = nlp_trf.meta["version"]
print(f"Modelo TRF ({version_trf}) cargado con éxito. Componentes: {nlp_trf.pipe_names}")

# ------------------------------------------------------------------------------
# Parámetros temporales
# ------------------------------------------------------------------------------
HIGH_RES_FS = 100  # Resolución de la matriz intermedia (100 Hz = 10 ms)

Cargando modelo de spaCy: en_core_web_trf...
✅ Modelo cargado exitosamente.

Modelos disponibles en spaCy:
Modelo SM (3.8.0) cargado con éxito. Componentes: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Modelo TRF (3.8.0) cargado con éxito. Componentes: ['transformer', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [8]:
# ==============================================================================
# Cell 2: EDA of Sentence Boundaries and Orthographic Tokens
# ==============================================================================

"""
Script de Análisis Exploratorio de Datos (EDA) para las ventanas sintácticas.
Utiliza las pausas/silencios como delimitadores para segmentar el texto y 
extrae métricas estadísticas detalladas (cuartiles, extremos) y vistazos 
cualitativos del corpus, asegurando la viabilidad del análisis con spaCy.
"""

def check_spacy_limits(nlp_model: spacy.language.Language) -> None:
    """Verifica e imprime los límites configurados en el modelo cargado."""
    print("-" * 60)
    print(f"🔍 VERIFICACIÓN DE LÍMITES - Modelo: {nlp_model.meta['name']}")
    print(f"  -> Límite global de caracteres: {nlp_model.max_length:,}")
    if "trf" in nlp_model.meta["name"]:
        print("  -> Arquitectura: Transformer (Límite interno 512 tokens).")
    else:
        print("  -> Arquitectura: CNN/Estadística.")
    print("-" * 60)

def exploratory_data_analysis_chunks(base_dir: Path) -> None:
    """
    Realiza un EDA sobre los fragmentos de habla (chunks) delimitados por silencios.
    """
    textgrid_files = list(base_dir.rglob("*.TextGrid"))
    word_inventory = Counter()
    
    # Aquí guardaremos toda la data de los fragmentos para análisis estadístico
    chunks_data = []
    
    for file_path in textgrid_files:
        try:
            tg = tgt.io.read_textgrid(str(file_path), include_empty_intervals=True)
            word_tier = None
            
            for name in tg.get_tier_names():
                if 'word' in name.lower():
                    word_tier = tg.get_tier_by_name(name)
                    break
                    
            if word_tier is not None:
                current_chunk_words = []
                
                for interval in word_tier.intervals:
                    token = interval.text.strip()
                    
                    if token == "":
                        token = "<EMPTY_SILENCE>"
                    
                    word_inventory[token] += 1
                    
                    # Criterio empírico de pausa/separador
                    is_pause = (token == "<EMPTY_SILENCE>" or 
                                token.lower() in ['sp', 'spn', 'sil', 'br', 'lg'] or
                                '[' in token or '{' in token)
                    
                    if is_pause:
                        if len(current_chunk_words) > 0:
                            chunks_data.append({
                                'story': file_path.name,
                                'length': len(current_chunk_words),
                                'text_glimpse': " ".join(current_chunk_words)
                            })
                            current_chunk_words = []
                    else:
                        current_chunk_words.append(token)
                        
                # Añadir el último chunk si la historia terminó sin silencio final
                if len(current_chunk_words) > 0:
                    chunks_data.append({
                        'story': file_path.name,
                        'length': len(current_chunk_words),
                        'text_glimpse': " ".join(current_chunk_words)
                    })
                    
        except Exception:
            pass

    # 1. Etiquetas de Ruido
    df_audit = pd.DataFrame.from_dict(word_inventory, orient='index', columns=['frequency'])
    noise_suspects = df_audit[
        df_audit.index.str.contains(r'\[|\]|\{|\}|\<|\>|spn|^sp$|^sil$|^br$|^lg$', regex=True, na=False)
    ]
    print("\n1. ETIQUETAS DE PAUSA/RUIDO MÁS FRECUENTES:")
    display(noise_suspects.sort_values('frequency', ascending=False).head(10))
    
    # 2. Análisis Estadístico de Fragmentos (EDA)
    df_chunks = pd.DataFrame(chunks_data)
    
    print("\n2. ANÁLISIS ESTADÍSTICO DE VENTANAS (CHUNKS):")
    if not df_chunks.empty:
        # Imprime la tabla estadística (count, mean, std, min, 25%, 50%, 75%, max)
        display(pd.DataFrame(df_chunks['length'].describe()).T)
        
        # Opciones de visualización para ver el texto completo
        pd.set_option('display.max_colwidth', 150)
        
        print("\nTOP 5 FRAGMENTOS MÁS CORTOS:")
        display(df_chunks.nsmallest(5, 'length')[['story', 'length', 'text_glimpse']])
        
        print("\nTOP 5 FRAGMENTOS MÁS LARGOS:")
        display(df_chunks.nlargest(5, 'length')[['story', 'length', 'text_glimpse']])
        
        # Restaurar visualización
        pd.reset_option('display.max_colwidth')
        
    else:
        print("No se pudieron extraer fragmentos de los archivos.")

# Ejecutar verificación y EDA
check_spacy_limits(nlp)
exploratory_data_analysis_chunks(BASE_DIR)

------------------------------------------------------------
🔍 VERIFICACIÓN DE LÍMITES - Modelo: core_web_trf
  -> Límite global de caracteres: 1,000,000
  -> Arquitectura: Transformer (Límite interno 512 tokens).
------------------------------------------------------------

1. ETIQUETAS DE PAUSA/RUIDO MÁS FRECUENTES:


,frequency
sp,23432
{BR},1828
{LG},1381
{NS},940
{LS},270
<EMPTY_SILENCE>,95
{CG},31
{SP},4
{NS,2
{IG},2



2. ANÁLISIS ESTADÍSTICO DE VENTANAS (CHUNKS):


,count,mean,std,min,25%,50%,75%,max
length,24530.0,6.26274,5.41019,1.0,2.0,5.0,8.0,65.0



TOP 5 FRAGMENTOS MÁS CORTOS:


,story,length,text_glimpse
2,odetostepfather.TextGrid,1,MY
3,odetostepfather.TextGrid,1,SECRET
11,odetostepfather.TextGrid,1,SO
13,odetostepfather.TextGrid,1,UM
15,odetostepfather.TextGrid,1,AND



TOP 5 FRAGMENTOS MÁS LARGOS:


,story,length,text_glimpse
6193,myfirstdaywiththeyankees.TextGrid,65,YOU KNOW EVEN AT THAT MOMENT I'D NEVER REALLY THOUGHT ABOUT THE EXPERIENCE IN THOSE TERMS AND HE COULD HAVE SAID SO MANY OTHER THINGS THAT WOULDN'...
14372,haveyoumethimyet.TextGrid,55,LIKE I WENT HOME FOR THANKSGIVING AND MY FAMILY WAS LIKE SO HAVE YOU MET HIM YET AND I WAS LIKE YEAH AND THEY WERE LIKE WHAT DID HE SAY AND I WAS ...
2966,life.TextGrid,52,SO SHE HAD THIS REALLY STRATEGIC LIST IT'S LIKE YOU INVITE JUDY AND SHE'S GONNA TELL ALL THE PEOPLE IN THE ARTS COMMUNITY THAT MY MOM WAS INVOLVED...
6259,myfirstdaywiththeyankees.TextGrid,52,OK THANKS I'll YOU KNOW I'M SORRY TO INTERRUPT I GO OFF AT AT THIS POINT I'M LIKE SPRINTING DOWN THE HALLWAYS LIKE THE TUNNELS BENEATH THE STANDS ...
21205,naked.TextGrid,49,AND SO I SIT THERE AND UM ALL OF A SUDDEN LIKE IT WAS A REALLY COLD NIGHT AND UM THE WIND BLEW IN AND I GOT REALLY COLD AND OF COURSE I HADN'T EAT...


In [ ]:
# ==============================================================================
# Cell 3: Lexical-Statistical Extraction Engine (Surface Form)
# ==============================================================================

"""
Genera la matriz temporal de características léxico-estadísticas (Ocurrencia, 
Frecuencia, Longitud y Duración) proyectada a 100 Hz.

JUSTIFICACIÓN METODOLÓGICA (Forma Superficial vs. Lema):
En el marco de los modelos predictivos de la señal BOLD (encoding models), 
la función de este espacio es 'controlar' (absorber) la varianza neurohemodinámica 
asociada con el esfuerzo de procesamiento ascendente (bottom-up) y el acceso léxico.

Se decide operar estrictamente sobre la 'forma superficial' (surface form) de la 
palabra pronunciada, y no sobre su lema, por la siguiente razón de control cruzado:
Las formas finitas que constituyen el fenómeno de interés ("pasado" vs "no pasado") 
tienen frecuencias de ocurrencia dispares en el uso real del inglés. Por ejemplo, 
"is" (no pasado) tiene mayor frecuencia que "was" (pasado). Si se calculara la 
frecuencia sobre el lema ("be" para ambos casos), el modelo asignaría el mismo 
peso de control a ambas palabras. Esto generaría un riesgo de confusión en la 
regresión regularizada: el modelo fMRI podría atribuir erróneamente al "Tiempo 
Gramatical" una varianza que en realidad pertenece a la diferencia acústica y de 
acceso léxico entre "is" y "was". 

Al extraer la frecuencia logarítmica (Zipf) de la forma superficial exacta, 
se purifica el Espacio de Interés, garantizando que el delta R^2 (ΔR^2) capturado 
por el tiempo gramatical refleje procesamiento morfosintáctico y no un mero 
efecto de frecuencia léxica.
"""

import re
from typing import Union
import numpy as np
import pandas as pd
import tgt
from pathlib import Path
from wordfreq import zipf_frequency

# Diccionario empírico de etiquetas de ruido/pausa (basado en el EDA).
NOISE_TAGS = {
    '<empty_silence>', 'sp', '{br}', '{lg}', '{ns}', 
    '{ls}', '{cg}', '{sp}', '{ns', '{ig}'
}

LEXICAL_FEATURE_NAMES = [
    'word_presence', 
    'lexical_frequency', 
    'word_length_chars', 
    'word_duration_secs'
]

def process_lexical_token(raw_token: str) -> dict:
    """
    Evalúa si un token es una palabra válida y calcula sus métricas léxicas
    basándose en su forma superficial exacta.
    
    Args:
        raw_token (str): Símbolo bruto extraído de la capa 'words'.
        
    Returns:
        dict: Métricas estadísticas. Si es ruido/pausa, retorna ceros.
    """
    token_lower = str(raw_token).strip().lower()
    
    # Identificar si es un silencio o marca metalingüística
    if not token_lower or token_lower in NOISE_TAGS:
        return {
            'word_presence': 0.0,
            'lexical_frequency': 0.0,
            'word_length_chars': 0.0
        }
        
    # Limpieza ortográfica de la forma superficial
    clean_word = re.sub(r'[^a-z\']', '', token_lower)
    
    if not clean_word:
        return {
            'word_presence': 0.0,
            'lexical_frequency': 0.0,
            'word_length_chars': 0.0
        }
        
    # Frecuencia Zipf de la forma superficial (aislando costo de acceso léxico real)
    freq_zipf = zipf_frequency(clean_word, 'en')
    
    return {
        'word_presence': 1.0,
        'lexical_frequency': freq_zipf,
        'word_length_chars': float(len(clean_word))
    }

def extract_lexical_statistical_space(textgrid_path: Union[str, Path], fs: int) -> pd.DataFrame:
    """
    Genera la matriz temporal léxico-estadística a alta resolución.
    
    Args:
        textgrid_path (Path o str): Ruta al archivo TextGrid.
        fs (int): Frecuencia de muestreo (100 Hz).
        
    Returns:
        pd.DataFrame: Matriz temporal de características (float32).
    """
    tg = tgt.io.read_textgrid(str(textgrid_path), include_empty_intervals=True)
    
    word_tier = None
    for name in tg.get_tier_names():
        if 'word' in name.lower():
            word_tier = tg.get_tier_by_name(name)
            break
            
    if word_tier is None:
        raise ValueError(f"Capa 'words' no encontrada en {textgrid_path}")
        
    total_duration = word_tier.end_time
    total_samples = int(np.ceil(total_duration * fs))
    
    feature_matrix = np.zeros((total_samples, len(LEXICAL_FEATURE_NAMES)), dtype=np.float32)
    
    for interval in word_tier.intervals:
        metrics = process_lexical_token(interval.text)
        
        start_idx = int(np.floor(interval.start_time * fs))
        end_idx = int(np.ceil(interval.end_time * fs))
        end_idx = min(end_idx, total_samples)
        
        interval_duration = interval.end_time - interval.start_time
        
        if metrics['word_presence'] > 0:
            features = [
                metrics['word_presence'],
                metrics['lexical_frequency'],
                metrics['word_length_chars'],
                interval_duration
            ]
            feature_matrix[start_idx:end_idx, :] = features

    time_axis = np.arange(total_samples) / fs
    
    df_features = pd.DataFrame(feature_matrix, columns=LEXICAL_FEATURE_NAMES, index=time_axis)
    df_features.index.name = 'time_seconds'
    
    return df_features

In [10]:
# ==============================================================================
# Cell 4: Execution and Lexical-Statistical Visualization
# ==============================================================================

"""
Prueba el motor de extracción tomando la primera historia válida del corpus 
y despliega en pantalla un fragmento de la matriz resultante para confirmar 
el comportamiento de las variables cuantitativas continuas.
"""

try:
    # Buscar el primer archivo válido (ignorando los problemáticos conocidos)
    textgrid_files = list(BASE_DIR.rglob("*.TextGrid"))
    valid_file = next((f for f in textgrid_files if f.name not in ['legacy.TextGrid', 'exorcism.TextGrid']), None)
            
    if valid_file:
        print(f"Generando espacio léxico-estadístico para: {valid_file.name} ...")
        
        # Ejecutar extracción
        df_lexical_space = extract_lexical_statistical_space(valid_file, HIGH_RES_FS)
        
        print("\nInformación del Espacio de Características:")
        print(f"  -> Dimensiones (Muestras Temporales x Rasgos): {df_lexical_space.shape}")
        print(f"  -> Tipo de dato: {df_lexical_space.values.dtype}\n")
        
        print("Muestra de los cambios en los atributos durante el habla activa:")
        
        # Filtramos instantes donde hay palabra y mostramos cambios (drop_duplicates)
        active_samples = df_lexical_space[df_lexical_space['word_presence'] > 0]
        display(active_samples.drop_duplicates().head(12))
        
    else:
        print("Error: No se encontró ningún archivo válido para procesar.")

except Exception as e:
    print(f"Ocurrió un error en la ejecución: {str(e)}")

Generando espacio léxico-estadístico para: odetostepfather.TextGrid ...

Información del Espacio de Características:
  -> Dimensiones (Muestras Temporales x Rasgos): (82811, 4)
  -> Tipo de dato: float32

Muestra de los cambios en los atributos durante el habla activa:


,word_presence,lexical_frequency,word_length_chars,word_duration_secs
time_seconds,,,,
0.03,1.0,5.73,5.0,0.189569
0.22,1.0,7.73,3.0,0.099773
0.32,1.0,4.78,9.0,0.548753
0.87,1.0,7.07,2.0,0.129705
1.00,1.0,6.14,3.0,0.109751
1.10,1.0,4.49,5.0,0.409070
1.51,1.0,5.03,7.0,0.379138
1.89,1.0,7.41,3.0,0.109751
2.00,1.0,4.26,2.0,0.179592
